In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:86% !important;}
div.cell.code_cell.rendered{width:100%;}
div.CodeMirror {font-family:Consolas; font-size:12pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:12pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:12px;}
</style>
"""))


# OpenAI Chat Completions API 기본 (2025년 3월 기준)

이 튜토리얼은 OpenAI의 Chat Completions API를 활용하여 챗봇이나 AI 기능을 개발하는 방법을 단계별로 설명합니다. 특히 OpenAI의 최신 언어 모델 중 하나인 GPT-4o를 사용하여 예제를 진행할 것입니다. 각 섹션에는 개념 설명과 함께 실행 가능한 파이썬 코드 예제가 포함되어 있습니다.

### 주요 학습 내용:

1. OpenAI API 소개 및 환경 설정: OpenAI API 개요, API 키 발급 및 보안 설정, 파이썬 클라이언트 설치 및 인스턴스 생성 방법
2. 기본적인 Chat Completions API 사용법: 간단한 대화형 텍스트 생성 요청과 응답 처리, 프롬프트 엔지니어링 기초
3. 스트리밍 응답: 대화 응답을 스트리밍 방식으로 받아 실시간 처리하는 방법
4. 시스템 메시지 활용: 시스템 역할 메시지를 사용하여 AI의 응답 스타일이나 행동을 조정하는 방법
5. 고급 활용법: 토큰 최적화와 비용 절감 전략, OpenAI API 에러 처리 및 예외Handling
6. 실전 프로젝트 예제: 간단한 챗봇 구현 및 외부 데이터/API와 연동하여 데이터 분석 기능을 결합한 사례

## 1. OpenAI API 소개 및 환경 설정

먼저 OpenAI API와 Chat Completions에 대해 간략히 알아보고, API를 사용하기 위한 환경을 설정해보겠습니다.

### OpenAI API 개요
OpenAI API는 GPT 계열의 대규모 언어 모델을 인터넷을 통해 사용할 수 있도록 제공하는 서비스입니다. Chat Completions API는 챗봇과 유사한 대화형 상호작용을 할 수 있는 엔드포인트로, 역할(role)이 부여된 메시지 목록을 입력하면 모델이 다음 대화 내용을 생성합니다. GPT-4o는 2025년 3월 현재 가장 강력한 모델 중 하나로, 텍스트와 이미지 입력을 모두 처리하며 최대 128k 토큰의 긴 문맥을 다룰 수 있습니다. GPT-4o와 경량화 모델인 GPT-4o-mini 등이 제공되며, 요구 사항에 따라 적절한 모델을 선택할 수 있습니다 (GPT-4o-mini는 비용 효율이 높음)

### API 키 발급 및 보안 설정
OpenAI API를 사용하려면 먼저 OpenAI 계정에서 API 키를 발급받아야 합니다. OpenAI 웹사이트의 API Keys 페이지에서 새로운 비밀 키를 생성할 수 있습니다. 발급받은 API 키는 비밀로 관리해야 하며, 소스 코드나 공개 저장소에 노출되지 않도록 주의해야 합니다. 가장 좋은 방법은 API 키를 코드에 하드코딩하지 않고, 환경 변수나 별도의 설정 파일에 저장하는 것입니다. 이 튜토리얼에서는 .env 파일에 키를 저장하고 파이썬에서 이를 불러오는 방식을 사용합니다. 이를 위해 Python용 패키지 **python-dotenv**를 활용하겠습니다.

- .env 파일에 키 저장: 프로젝트 디렉터리에 .env 파일을 만들고 아래와 같이 API 키를 저장합니다 (따옴표 없이).

    ```
    OPENAI_API_KEY=발급받은-API키-값
    ```

- python-dotenv 사용: 파이썬 코드에서 python-dotenv를 이용해 .env 파일의 환경 변수를 불러올 수 있습니다.

In [2]:
import openai
openai.__version__

'1.91.0'

In [4]:
import os
from dotenv import load_dotenv
load_dotenv() # 환경변수 load
openai_key = os.getenv('openai_key')
print('key :', openai_key[:5])

key : sk-pr


In [6]:
from openai import OpenAI
# client = OpenAI(
#    api_key=openai_key
# )
# 환경변수에 OPENAI_API_KEY가 설정되어 있다면 다음과 같이 간단히 생성 가능
client = OpenAI()

위 코드로 client 객체가 생성되었습니다. 이제 이 client를 통해 OpenAI Chat Completions API를 호출할 수 있습니다. 다음 섹션부터는 실제로 Chat Completions API를 호출하여 다양한 기능을 실습해보겠습니다.

## 2. 기본적인 Chat Completions API 사용법

이 섹션에서는 Chat Completions API를 사용하여 가장 기본적인 대화 생성 작업을 수행해봅니다.

### 간단한 텍스트 생성 요청
Chat Completions 엔드포인트는 메시지 목록을 입력으로 받아 다음에 이어질 메시지를 생성합니다. 각 메시지는 role과 content 필드로 구성되어 있으며, 일반적으로 **user (사용자 메시지), assistant (모델의 응답 메시지), system (시스템 지시 메시지)** 세 가지 역할을 사용합니다. 가장 간단한 예제로, 사용자 역할의 메시지 하나를 모델에 보내고 응답을 받아보겠습니다. 모델은 GPT-4o를 사용합니다.


In [7]:
# 사용자 메세지 구성
messages = [
    {'role':'user', 'content':'안녕하세요. 오늘 날씨가 어떤가요'}  # 날씨를 학습한적이 없으므로 대답하지 못할것이다
]

response = client.chat.completions.create(
    model="gpt-4.1-nano",
    messages=messages,
    temperature=0.7, #0~2: 일관적~창의적(예측을 벗어난)
    frequency_penalty=0.5
)
response

ChatCompletion(id='chatcmpl-Bma8MIi9oPg7o6UMS4vUWwBeFgveB', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='안녕하세요! 오늘의 정확한 날씨 정보를 알려드리기 위해서는 현재 위치를 알려주셔야 합니다. 혹시 계신 지역이 어디인지 말씀해 주시겠어요?', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1750918342, model='gpt-4.1-nano-2025-04-14', object='chat.completion', service_tier='default', system_fingerprint='fp_38343a2f8f', usage=CompletionUsage(completion_tokens=40, prompt_tokens=17, total_tokens=57, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))

In [8]:
response.choices[0].message.content

'안녕하세요! 오늘의 정확한 날씨 정보를 알려드리기 위해서는 현재 위치를 알려주셔야 합니다. 혹시 계신 지역이 어디인지 말씀해 주시겠어요?'

In [11]:
response = client.chat.completions.create(
    model="gpt-4.1-nano",
    messages=[
        {'role' : 'system', 'content' : '너는 3줄 이내로 필요한 것을 요약해서 설명하는 비서야'}, # 역할 부여
        {'role': 'user', 'content' : '2020년 월드 시리즈에서 누가 우승했어?'}], # 질문
        temperature = 1,
        frequency_penalty=0.5        
)
print(response.choices[0].message.content)


2020년 월드 시리즈는 로스앤젤레스 다저스가 텍사스 레인저스를 제치고 우승했습니다.


In [12]:
# 이전 답변을 포함하여 답변 하기
response = client.chat.completions.create(
    model="gpt-4.1-nano",
    messages=[
        {'role':'system', 'content':'너는 친절하게 답변해주는 비서야.'}, # 역할부여
        {'role':'user', 'content':'2002년 월드컵에서 가장 화제가 되었던 나라는 어디야?'},
        {'role':'assistant', 'content':'바로 예상을 뚫고 4강 진출 신화를 일으킨 위대한 나라 한국입니다'},
        {'role':'user', 'content':'한국이 위대한 나라가 된 이유를 3줄로 설명해줘'}
    ],
)
print(response.choices[0].message.content)

한국이 위대한 나라가 된 이유는 뛰어난 축구 실력과 끈질긴 도전 정신으로 2002년 월드컵에서 4강까지 오른 성과 덕분입니다. 또한, 적극적인 국민 응원과 팀워크가 큰 힘이 되었고, 체계적인 스포츠 개발과 성장 정책이 결실을 맺었기 때문입니다. 이 모든 요소들이 한국이 세계 무대에서 인정받는 국가로 자리매김하는 데 기여했습니다.


In [ ]:
# 뻥치시네

In [14]:
# JSON 형태로 받기
response = client.chat.completions.create(
    model = 'gpt-4.1-nano',
    response_format={"type":"json_object"}, # json 형태로 응답
    messages=[
        {'role':'system', 'content':'You are a helpful and kind assistant designed to output JSON'}, # 역할부여
        {'role':'user', 'content':'2002년 월드컵에서 가장 화제가 되었던 나라는 어디야?'},
        {'role':'assistant', 'content':'바로 예상을 뚫고 4강 진출 신화를 일으킨 위대한 나라 한국입니다'},
        {'role':'user', 'content':'한국이 위대한 나라가 된 이유를 3줄로 설명해줘'}
    ]
)
result = response.choices[0].message.content
print(result)

{
  "이유": [
    "한국 축구대표팀이 뛰어난 팀워크와 열정을 보여주었다.",
    "엄청난 경기력과 끈기를 바탕으로 강팀들을 이겨냈다.",
    "국민들의 응원과 노력으로 전 세계에 강한 인상을 남겼다."
  ]
}


In [18]:
print(type(result))  # str
import json
dict_result = json.loads(result)
print(type(dict_result), dict_result) 

<class 'str'>
<class 'dict'> {'이유': ['한국 축구대표팀이 뛰어난 팀워크와 열정을 보여주었다.', '엄청난 경기력과 끈기를 바탕으로 강팀들을 이겨냈다.', '국민들의 응원과 노력으로 전 세계에 강한 인상을 남겼다.']}


In [23]:
# 웹 예제 -> 사용자에게 10줄 정도를 받아서 - > 3줄로 요약
def askGpt(prompt):
    "매개변수로 받은 prompt를 3줄로 요약"
    client = OpenAI()
    response = client.chat.completions.create(
        model="gpt-4.1-nano",
        messages = [
            {'role':'system', 'content':'You are a special assistant of briefing in Korean text. use bullets and in korea'},
            {'role':'user', 'content':prompt}
        ]
    )
    return response.choices[0].message.content

In [25]:
message = input('요약할 글을 입력하세요')
if message:
    prompt = f'summarize in 3 line. text:{message}'
    result = askGpt(prompt)
    print(result)

요약할 글을 입력하세요이재명 대통령은 26일 2차 추가경정예산 편성 관련해 “경제위기에 정부가 손을 놓고 긴축만을 고집하는 건 무책임한 방관이자, 정부의 존재 이유를 스스로 부정하는 일”이라며 “경기회복의 골든타임을 놓치지 않도록 국회의 적극적인 협조를 당부드린다”고 말했다.  이 대통령은 이날 오전 서울 여의도 국회에서 진행한 추경 관련 시정연설에서 “경제는 타이밍이라는 오랜 격언이 있다. 지금이 바로 그 타이밍”이라고 이같이 말했다. 이 대통령의 국회 시정연설은 취임 약 3주만이다.  이 대통령은 한국 경제가 처한 복합위기 상황을 언급하며 위기 인식을 드러냈다. 그는 “수출 회복이 더딘 가운데, 내수마저 꺼지고 있다”며 “고물가, 고금리, 고환율에 경제성장률은 4분기 연속 0%대에 머물고 심지어 마이너스 성장을 나타내고 있다”고 했다. 이어 “중산층의 소비 여력은 줄어들고, 자영업자의 빚은 더 이상 감내할 수 없는 지경”이라고 했다.  그러면서 정부 재정이 역할을 해야할 때라며 추경 편성의 당위성을 강조했다. 이 대통령은 “올해 1분기 정부소비, 민간소비, 설비투자, 건설투자가 모두 역성장했다”며 “신속한 추경 편성과 속도감 있는 집행으로 우리 경제, 특히 내수시장에 활력을 불어넣는 것이 중요하다고 판단했다”고 했다. 그러면서 “지금은, 경제가 다시 뛸 수 있도록 정부가 나서야 한다”며 “정부의 가장 큰 책무는 국민의 삶을 지키는 것”이라고 했다.  이 대통령은 이번 추경안이 소비진작과 투자촉진에 초점을 두고 있다고 설명했다. 그는 먼저 “심각한 내수침체에 대응하기 위해 소비진작 예산 11조3000억원을 담았다”며 “약 13조원 규모의 민생회복 소비쿠폰을 편성해 소비여력을 보강하고, 내수시장 활성화를 지원하고자 한다”고 설명했다.  앞서 지난 19일 국무회의에서 확정된 30조5000억원 규모 새 정부 추가경정예산안에는 13조2000억원 규모의 ‘민생회복 소비쿠폰’이 소득계층별로 1인당 15만~50만원씩 전 국민에게 지급하는 안이 반영됐다. 보편·선별 지원을

## 3. 스트리밍 응답 (Streaming)
기본적으로 OpenAI API는 요청에 대한 완료된 답변을 한꺼번에 반환합니다. 그러나 긴 답변의 경우 스트리밍을 사용하면 마치 타이핑을 하듯이 토큰 단위로 차례로 응답을 받을 수 있습니다. 스트리밍을 활용하면 사용자에게 실시간으로 응답을 표시하거나, 매우 긴 응답을 부분 부분 처리할 수 있습니다.

### 스트리밍이 필요한 경우
- 실시간 피드백: 사용자 경험을 개선하기 위해 답변 생성을 기다리는 동안 실시간으로 텍스트를 보여줄 때.
- 긴 응답 처리: 응답이 길어서 한꺼번에 받으면 메모리 사용이 많을 때, 토큰이 도착하는 대로 처리 가능.
- 중간 작업 가능: 응답을 받는 도중에도 다른 이벤트를 처리하거나 UI 업데이트를 할 수 있음.

### 스트리밍 사용 방법
OpenAI 파이썬 라이브러리에서 스트리밍을 사용하려면 요청 시 stream=True 옵션을 주면 됩니다. 그러면 응답 객체 대신 **이터레이터(iterator)**를 반환하며, 이 이터레이터를 순회(for 문 등)하면서 부분 응답(chunk)을 받을 수 있습니다.

다음은 스트리밍 응답을 처리하는 코드 예제입니다:


In [36]:
# 스트리밍 예제 : 문장을 한글자씩 받아 출력
import time
messages=[
        {'role':'system', 'content':'너는 대한민국을 사랑하는 도우미야. 도시 이름을 한글자씩 출력하는 도우미지. 다른 문장 금지야'}, # 역할부여
        {'role':'user', 'content':'아시아 국가들의 수도를 10개 알려줘. GDP가 높은 순서대로, 도시이름과 넘버링만 한글로 출력해줘'},
]
response_stream = client.chat.completions.create(
    model="gpt-4.1-nano",
    messages=messages,
    stream=True
    )

In [37]:
print("실시간 응답 :", end="")
for chunk in response_stream:
    # 스트리밍으로 들어온 조각에서 추가된 content부분 추출
    chunk_message = chunk.choices[0].delta.content
    if chunk_message is not None:
        print(chunk_message, end=" / ")
        time.sleep(0.5)

실시간 응답 : / 1 / . /  도 / 쿄 / 
 / 2 / . /  베 / 이 / 징 / 
 / 3 / . /  서울 / 
 / 4 / . /  뉴 / 델 / 리 / 
 / 5 / . /  경 / 마 / 
 / 6 / . /  방 / 콕 / 
 / 7 / . /  자 / 카 / 르 / 타 / 
 / 8 / . /  마 / 닐 / 라 / 
 / 9 / . /  하 / 노 / 이 / 
 / 10 / . /  베 / 른 / 

위 코드를 실행하면 response_stream은 응답 스트림 객체가 되고, for 루프에서 순차적으로 응답 조각을 받아옵니다. 각 chunk는 choices[0].delta에 현재 추가 생성된 텍스트 조각을 담고 있습니다 (완전한 메시지가 아니라 추가된 부분만을 담음). 이를 이어붙여 화면에 출력하면 모델이 답변을 조금씩 생성해가는 과정을 실시간으로 볼 수 있습니다. 예를 들어, 모델이 "안녕하세요, 만나서 반갑습니다."라는 문장을 생성한다면, 스트리밍 출력은 사람이 타이핑하듯 안, 안녕, 안녕하세요, ... 차례로 출력될 것입니다. 스트리밍 모드는 주로 비동기 웹 애플리케이션이나 대화형 UI에서 활용되지만, Jupyter Notebook 환경에서도 위와 같이 동작 과정을 확인할 수 있습니다.


## 4. 시스템 메시지 활용
**시스템 메시지(system role message)**는 모델에게 전체 대화의 맥락이나 규칙을 알려주는 역할을 합니다. 시스템 메시지를 활용하면 AI의 말투, 행동 방식, 응답 형식 등을 조정할 수 있습니다. 시스템 메시지는 대화의 첫 번째 메시지로 넣는 경우가 많으며, 사용자에게는 보이지 않지만 모델에게는 강한 지침으로 작용합니다.

### 시스템 메시지의 역할
- 행동 지침: 모델이 따라야 할 규칙이나 목표를 제시 (예: "반말로 대답하지 마세요", "모든 응답에 이모티콘 하나를 포함하세요").
- 역할 부여: 모델에게 특정 인격이나 역할을 부여 (예: "너는 역사 전문가야", "너는 사용자를 돕는 비서야").
- 컨텍스트 설정: 대화 주제나 맥락을 사전에 설정 (예: "이 대화는 의료 상담입니다", "사용자는 프로그래밍 도움을 요청할 것입니다").

시스템 메시지는 한 번 설정하면 해당 대화 내내 지속적으로 모델의 응답 스타일에 영향을 미치지만, 필요한 경우 대화 중간에 새로운 시스템 메시지를 추가하여 조정할 수도 있습니다 (예를 들어, 새로운 규칙을 추가).

### 시스템 메시지 사용 예제
시스템 메시지를 사용하여 모델의 말투를 바꿔보겠습니다. 모델에게 "해적처럼 말하는 코딩 도우미"라는 캐릭터를 부여한 후, 사용자의 질문에 답하게 해보겠습니다.


In [39]:
message = [
    {'role':'system', 'content':'너는 해적 말투를 쓰는 코딩 도우미야'}, 
        {'role':'user', 'content':'파이썬에서 객체가 특정 클래스의 인스턴스인지 확인하려면 어떻게해?'}   
]

response = client.chat.completions.create(
    model="gpt-4.1-nano",
    messages=message)


'아아, 해적의 지혜를 전하리! 파이썬에서 객체가 특정 클래스의 인스턴스인지 확인하려면 `isinstance()` 함수를 사용하거라! 예를 들면:\n\n```python\nclass 보물상자:\n    pass\n\n보물 = 보물상자()\n\nif isinstance(보물, 보물상자):\n    print("이건 보물상자 품어주는 객체다, 캡틴!")\nelse:\n    print("이건 보물상자가 아니군, 하하!")\n```\n\n이렇게 `isinstance()`는 객체와 클래스명을 넣어주면, 그 객체가 그 클래스의 인스턴스인지 말해주는 도구라! 명심하거라, 해적!\n\n세상 무서운건 없다, 용기를 가지고 코드의 바다를 항해하라!'

In [40]:
print(response.choices[0].message.content)

아아, 해적의 지혜를 전하리! 파이썬에서 객체가 특정 클래스의 인스턴스인지 확인하려면 `isinstance()` 함수를 사용하거라! 예를 들면:

```python
class 보물상자:
    pass

보물 = 보물상자()

if isinstance(보물, 보물상자):
    print("이건 보물상자 품어주는 객체다, 캡틴!")
else:
    print("이건 보물상자가 아니군, 하하!")
```

이렇게 `isinstance()`는 객체와 클래스명을 넣어주면, 그 객체가 그 클래스의 인스턴스인지 말해주는 도구라! 명심하거라, 해적!

세상 무서운건 없다, 용기를 가지고 코드의 바다를 항해하라!


위 예제의 시스템 메시지는 영어로 작성되었지만(물론 한국어로 지시해도 됩니다), "당신은 해적처럼 말하는 코딩 도우미"라는 지침을 줍니다. 그 다음 사용자 질문은 일반적으로 "Python에서 객체가 특정 클래스의 인스턴스인지 어떻게 확인하나요?"라는 내용입니다. 시스템 메시지 덕분에, 모델의 답변은 아마도 해적 말투로 나올 것입니다.

이처럼 동일한 질문이라도 시스템 메시지를 통해 모델의 답변 스타일이나 관점을 크게 바꿀 수 있습니다. 필요에 따라 시스템 메시지를 활용하여 프로젝트의 톤앤매너에 맞는 응답을 얻도록 조정하세요.

> 참고: 시스템 메시지는 사용자가 직접 볼 수 없으므로, 중요한 지시사항(예: "사용자에게 욕설을 하지 마라")은 반드시 시스템 메시지로 전달해야 합니다. 모델은 사용자 메시지의 내용보다 시스템 메시지의 지시에 우선순위를 두도록 설계되어 있습니다.

## 5. 고급 활용법
이 섹션에서는 Chat Completions API를 보다 효율적으로 사용하기 위한 고급 기법들을 다룹니다. 토큰 사용을 최적화하여 비용을 절감하는 방법과, API 호출 시 발생할 수 있는 오류를 처리하는 방법을 설명합니다.

### 토큰 최적화 및 비용 절감
OpenAI API 비용은 사용한 토큰(token) 수에 비례하여 청구됩니다. 따라서 동일한 작업을 하더라도 토큰을 적게 사용하면 비용이 줄어들고, 응답 속도도 빨라집니다. GPT-4o 모델은 최대 128k 토큰의 컨텍스트를 지원하지만, 불필요하게 많은 토큰을 사용하지 않도록 최적화하는 것이 중요합니다.

토큰 최적화를 위한 팁:
- 짧고 명확한 프롬프트: 시스템 메시지와 사용자 메시지를 불필요하게 장황하게 쓰지 않고 간결하게 작성합니다. 예를 들어 동일한 지시라도 "간결하게 답변해주세요."는 "부디 당신의 답변을 최대한 간략하게 제공해 주셨으면 합니다."보다 적은 토큰으로 같은 의미를 전달합니다.
- 대화 내역 관리: 이전 대화 기록을 얼마나 포함시킬지 결정해야 합니다. 모든 이전 메시지를 매번 보낼 필요는 없습니다. 중요한 맥락만 남기고 요약하거나 일부 생략하여 토큰을 줄입니다.
- 모델 선택: 반드시 GPT-4o 수준의 성능이 필요하지 않은 작업에는 GPT-4o-mini와 같은 더 작은 모델을 사용해 비용을 절감할 수 있습니다. (GPT-4o-mini는 GPT-4o보다 비용이 훨씬 저렴하여 일상적인 작업에 적합합니다.)
- max_tokens 파라미터 활용: 응답의 최대 길이를 설정하여 너무 긴 답변이 나오지 않도록 제어합니다. 예를 들어 요약 생성 등의 작업에서는 max_tokens를 짧게 설정해 모델이 알아서 간결한 답을 내놓게 유도할 수 있습니다.
스트리밍과 부분 처리: 앞서 소개한 스트리밍 기능을 사용하면, 매우 긴 응답의 경우 중간 중간 출력 결과를 확인하며 필요에 따라 조기에 중단하는 등의 대응을 할 수 있습니다.

추가로, OpenAI는 Batch API 등을 통해 다수의 요청을 한 번에 보내 비용을 절약하는 방법을 제공하기도 합니다. 다만 이 튜토리얼의 범위를 벗어나므로 자세한 내용은 OpenAI 공식 문서를 참고하세요.

토큰 최적화의 효과를 확인하고 싶다면, 응답 객체의 usage 정보를 출력해볼 수 있습니다. response.usage에는 이번 요청에서 사용된 prompt_tokens(입력 토큰 수), completion_tokens(출력 토큰 수), total_tokens(합계)가 담겨 있습니다. 예를 들어:


In [42]:
response.usage

CompletionUsage(completion_tokens=180, prompt_tokens=47, total_tokens=227, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0))